In [0]:
## KitKi Chim
## KC3949
## Applied DATA

In [0]:
from pyspark.sql.functions import col, current_date, floor, months_between
from pyspark.sql.functions import avg

In [0]:
# Load datasets
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)

In [0]:
# Print schema
print("=== DRIVERS ===")
df_drivers.printSchema()

print("=== RACES ===")
df_races.printSchema()

print("=== PIT STOPS ===")
df_pit_stops.printSchema()

print("=== RESULTS ===")
df_results.printSchema()

---
## Question 1 — Average, Slowest, and Fastest Pit Stop per Driver per Race [10 pts]

### Logic
A driver may make multiple pit stops during a single race. To find the **average**, **slowest** (max duration), and **fastest** (min duration) pit stop time for each driver in each race, I need to:
1. Join `pit_stops` with `drivers` (to get driver names) and `races` (to get race names).
2. Group by `raceId` and `driverId` (plus their readable names).
3. Compute `avg`, `max`, and `min` of the `milliseconds` column (which records each pit stop duration in ms).
4. Convert milliseconds to seconds for readability.
5. Order by race and driver for clarity.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Join pit_stops → drivers → races
df_q1 = (
    df_pit_stops
    .join(df_drivers, on="driverId", how="inner")
    .join(df_races, on="raceId", how="inner")
)

# Aggregate: average, max (slowest), min (fastest) pit stop per driver per race
df_q1_result = (
    df_q1
    .groupBy(
        df_races["raceId"],
        df_races["name"].alias("race_name"),
        df_races["year"].alias("race_year"),
        df_drivers["driverId"],
        F.concat_ws(" ", df_drivers["forename"], df_drivers["surname"]).alias("driver_name")
    )
    .agg(
        F.round(F.avg("milliseconds") / 1000, 3).alias("avg_pit_stop_sec"),
        F.round(F.max("milliseconds") / 1000, 3).alias("slowest_pit_stop_sec"),
        F.round(F.min("milliseconds") / 1000, 3).alias("fastest_pit_stop_sec"),
        F.count("stop").alias("num_pit_stops")
    )
    .orderBy("raceId", "driverId")
)

df_q1_result.show(20, truncate=False)


### Code Explanation — Q1

1. **`df_pit_stops.join(df_drivers, on="driverId")`** joins pit stop records with driver details so I can display driver names alongside the statistics.
2. **`.join(df_races, on="raceId")`** further joins race information to get the race name and year.
3. **`.groupBy(raceId, race_name, race_year, driverId, driver_name)`** groups all pit stop entries for the same driver within the same race.
4. **`F.avg("milliseconds") / 1000`** computes the mean pit stop duration and converts from milliseconds to seconds; `F.max(...)` and `F.min(...)` find the slowest and fastest stops respectively.
5. **`F.round(..., 3)`** rounds all results to 3 decimal places.
6. **`F.count("stop")`** counts how many pit stops the driver made in that race, giving useful context.
7. **`.orderBy("raceId", "driverId")`** sorts the output for easy reading.

### Extra Credit — Alternative Approach (Spark SQL)
An equivalent approach is to register the DataFrames as temp views and write the same logic in Spark SQL:

In [0]:
# EXTRA CREDIT: Spark SQL alternative for Q1
df_pit_stops.createOrReplaceTempView("pit_stops")
df_drivers.createOrReplaceTempView("drivers")
df_races.createOrReplaceTempView("races")
df_results.createOrReplaceTempView("results")

df_q1_sql = spark.sql("""
    SELECT
        r.raceId,
        r.name AS race_name,
        r.year AS race_year,
        d.driverId,
        CONCAT(d.forename, ' ', d.surname) AS driver_name,
        ROUND(AVG(p.milliseconds) / 1000, 3) AS avg_pit_stop_sec,
        ROUND(MAX(p.milliseconds) / 1000, 3) AS slowest_pit_stop_sec,
        ROUND(MIN(p.milliseconds) / 1000, 3) AS fastest_pit_stop_sec,
        COUNT(p.stop) AS num_pit_stops
    FROM pit_stops p
    JOIN drivers d ON p.driverId = d.driverId
    JOIN races r   ON p.raceId   = r.raceId
    GROUP BY r.raceId, r.name, r.year, d.driverId, d.forename, d.surname
    ORDER BY r.raceId, d.driverId
""")

df_q1_sql.show(10, truncate=False)


---
## Question 2 — Rank Average Pit Stop Time by Finishing Position per Race [20 pts]

### Logic
The question asks: for each finishing position in each race, what is the average pit stop time? Then rank-order by finishing position.
1. Compute the average pit stop time per driver per race from `pit_stops`.
2. Join with `results` to get each driver's finishing position (`positionOrder`).
3. Join with `races` and `drivers` for readable names.
4. Rank (order) within each race by finishing position using a window function on `positionOrder`.

Since each (raceId, driverId) pair in results has exactly one finishing position, I get one row per driver per race with both their avg pit time and their finishing rank.


In [0]:
# Step 1: average pit stop time per driver per race
df_driver_pit_avg = (
    df_pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.round(F.avg("milliseconds") / 1000, 3).alias("avg_pit_stop_sec"))
)

# Step 2: join with results to get finishing position, then races and drivers for names
df_q2 = (
    df_driver_pit_avg
    .join(df_results.select("raceId", "driverId", "positionOrder"), on=["raceId", "driverId"], how="inner")
    .join(df_races.select("raceId", F.col("name").alias("race_name"), "year"), on="raceId", how="inner")
    .join(
        df_drivers.select(
            "driverId",
            F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("driver_name")
        ),
        on="driverId",
        how="inner"
    )
)

# Step 3: rank by finishing position within each race
w_q2 = Window.partitionBy("raceId").orderBy("positionOrder")

df_q2_result = (
    df_q2
    .withColumn("finish_rank", F.row_number().over(w_q2))
    .select("raceId", "race_name", "year", "positionOrder", "driver_name", "avg_pit_stop_sec", "finish_rank")
    .orderBy("raceId", "positionOrder")
)

df_q2_result.show(30, truncate=False)


### Code Explanation — Q2

1. **`df_pit_stops.groupBy("raceId", "driverId").agg(F.avg("milliseconds"))`** first computes the average pit stop duration for each driver in each race, giving us one row per driver per race.
2. **`.join(df_results..., on=["raceId", "driverId"])`** brings in `positionOrder` (the official finishing position) from the results table.
3. **`Window.partitionBy("raceId").orderBy("positionOrder")`** defines a window that resets for each race and orders rows by finishing position.
4. **`F.row_number().over(w_q2)`** assigns a rank within each race window — this is equivalent to the rank order by finishing position.
5. The final `.orderBy("raceId", "positionOrder")` sorts the output so I can read each race's results from P1 downward.


### Extra Credit — Alternative: Aggregate by Finishing Position Across All Races
Instead of showing each driver individually, I can aggregate to see the average pit stop time per finishing position bucket across **all** races — revealing whether higher finishers tend to have shorter pit stops:

In [0]:
# EXTRA CREDIT: average pit stop time by finishing position across ALL races
df_q2_ec = (
    df_driver_pit_avg
    .join(df_results.select("raceId", "driverId", "positionOrder"), on=["raceId", "driverId"], how="inner")
    .groupBy("positionOrder")
    .agg(
        F.round(F.avg("avg_pit_stop_sec"), 3).alias("overall_avg_pit_sec"),
        F.count("*").alias("sample_size")
    )
    .orderBy("positionOrder")
)

df_q2_ec.show(25, truncate=False)


---
## Question 3 — Insert Missing Driver Codes [20 pts]

### Logic
Some drivers in the `drivers` table have a `NULL` or `\N` value in the `code` column. The standard F1 driver code is the **first three letters of the driver's surname, uppercased** (e.g., "Alonso" → "ALO", "Hamilton" → "HAM").

My approach:
1. Identify rows where `code` is null or `\N`.
2. Generate the code as `UPPER(SUBSTRING(surname, 1, 3))`.
3. Use `WHEN ... THEN ... OTHERWISE` to fill in missing codes while preserving existing ones.
4. Handle potential **duplicates** (two drivers with the same first 3 surname letters) — flag them so I know the generated code may need manual review.


In [0]:
# Check how many drivers have missing codes
df_drivers.filter(
    (F.col("code") == "\\N") | (F.col("code").isNull())
).select("driverId", "forename", "surname", "code").show(20, truncate=False)


In [0]:
# Generate missing codes: first 3 letters of surname, uppercased
df_q3_result = (
    df_drivers
    .withColumn(
        "code_filled",
        F.when(
            (F.col("code") == "\\N") | (F.col("code").isNull()),
            F.upper(F.substring(F.col("surname"), 1, 3))
        ).otherwise(F.col("code"))
    )
)

df_q3_result.select("driverId", "forename", "surname", "code", "code_filled").show(30, truncate=False)


In [0]:
# Check for duplicate codes after filling
df_q3_dupes = (
    df_q3_result
    .groupBy("code_filled")
    .agg(
        F.count("*").alias("count"),
        F.collect_list(F.concat_ws(" ", "forename", "surname")).alias("drivers")
    )
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

print("=== Duplicate codes after filling ===")
df_q3_dupes.show(20, truncate=False)


### Code Explanation — Q3

1. **`F.col("code") == "\\N"` and `F.col("code").isNull()`** identify drivers with missing codes. The CSV uses the literal string `\N` as a null marker, so I check for both conditions.
2. **`F.upper(F.substring(F.col("surname"), 1, 3))`** extracts the first 3 characters of the surname and converts to uppercase. `F.substring(col, 1, 3)` is 1-indexed in Spark.
3. **`F.when(...).otherwise(F.col("code"))`** applies the generated code only when the original is missing; existing codes are kept unchanged.
4. The **duplicate check** (`groupBy("code_filled").agg(count, collect_list)`) identifies any code collisions, which is important because the 3-letter rule can produce identical codes for drivers with similar surnames (e.g., "Schumacher" and "Scheckter" both → "SCH").


### Extra Credit — Alternative: Use Forename Initial to Disambiguate Duplicates

In [0]:
# EXTRA CREDIT: disambiguate by using first letter of forename for duplicates
# Step 1: find which codes are duplicated
dup_codes = (
    df_q3_result
    .groupBy("code_filled")
    .count()
    .filter(F.col("count") > 1)
    .select("code_filled")
)

# Step 2: for duplicate codes, replace 3rd letter with first letter of forename
df_q3_ec = (
    df_q3_result
    .join(dup_codes, on="code_filled", how="left_semi")  # only duplicated rows
    .withColumn(
        "code_disambiguated",
        F.concat(
            F.substring(F.col("code_filled"), 1, 2),
            F.upper(F.substring(F.col("forename"), 1, 1))
        )
    )
    .select("driverId", "forename", "surname", "code", "code_filled", "code_disambiguated")
)

print("=== Disambiguated codes for collision cases ===")
df_q3_ec.show(20, truncate=False)


---
## Question 4 — Youngest and Oldest Driver in Each Race [20 pts]

### Logic
**Definition of "Age":** A driver's age at the time of a race is defined as the number of **full years** between the driver's date of birth (`dob`) and the race date (`date`). Specifically, I use `floor(datediff(race_date, dob) / 365.25)` to account for leap years.

Steps:
1. Join `results` with `drivers` (for `dob`) and `races` (for race `date`).
2. Compute `age` = floor of days between `dob` and race `date` divided by 365.25.
3. Use window functions with `min` and `max` over each race to identify youngest and oldest.
4. Filter to only the youngest and oldest in each race.


In [0]:
# Join results → drivers → races
df_q4_base = (
    df_results
    .select("raceId", "driverId")
    .join(df_drivers.select("driverId", "forename", "surname", "dob"), on="driverId", how="inner")
    .join(df_races.select("raceId", F.col("name").alias("race_name"), F.col("date").alias("race_date"), "year"), on="raceId", how="inner")
    .dropDuplicates(["raceId", "driverId"])
)

# Compute age
df_q4_base = df_q4_base.withColumn(
    "age",
    F.floor(F.datediff(F.col("race_date"), F.col("dob")) / 365.25).cast("int")
)

# Window to find min and max age per race
w_race = Window.partitionBy("raceId")

df_q4_tagged = (
    df_q4_base
    .withColumn("min_age_in_race", F.min("age").over(w_race))
    .withColumn("max_age_in_race", F.max("age").over(w_race))
    .withColumn(
        "label",
        F.when(F.col("age") == F.col("min_age_in_race"), "YOUNGEST")
         .when(F.col("age") == F.col("max_age_in_race"), "OLDEST")
    )
    .filter(F.col("label").isNotNull())
)

df_q4_result = (
    df_q4_tagged
    .select(
        "raceId", "race_name", "year",
        F.concat_ws(" ", "forename", "surname").alias("driver_name"),
        "dob", "race_date", "age", "label"
    )
    .orderBy("raceId", "label")
)

df_q4_result.show(30, truncate=False)


### Code Explanation — Q4

1. **`df_results.select("raceId", "driverId")`** gives us the set of drivers who participated in each race.
2. **`.join(df_drivers..., on="driverId")`** brings in `dob`; **`.join(df_races..., on="raceId")`** brings in the race `date`.
3. **`F.datediff(race_date, dob)`** computes the number of days between birth and race day. Dividing by `365.25` (to approximate leap years) and taking `F.floor(...)` gives us the age in whole years.
4. **`Window.partitionBy("raceId")`** scopes the `F.min("age")` and `F.max("age")` aggregations to each race.
5. **`F.when(age == min_age, "YOUNGEST").when(age == max_age, "OLDEST")`** labels the youngest and oldest drivers.
6. **`.filter(F.col("label").isNotNull())`** keeps only the youngest and oldest rows.


### Extra Credit — Alternative: Use `months_between` for More Precision

In [0]:
# EXTRA CREDIT: more precise age using months_between
df_q4_ec = (
    df_q4_base
    .drop("age")
    .withColumn(
        "age_precise",
        F.round(F.months_between(F.col("race_date"), F.col("dob")) / 12, 2)
    )
)

df_q4_ec.select(
    "raceId", "race_name",
    F.concat_ws(" ", "forename", "surname").alias("driver"),
    "dob", "race_date", "age_precise"
).orderBy("raceId", "age_precise").show(10, truncate=False)


---
## Question 5 — Cumulative Podiums per Driver at Each Race [20 pts]

### Logic
For any given race, I want to know how many **total wins (1st)**, **2nd places**, and **3rd places** each driver has accumulated **up to and including that race** (i.e., a running/cumulative count).

Steps:
1. From `results`, create indicator columns: `is_win` (positionOrder == 1), `is_2nd` (== 2), `is_3rd` (== 3).
2. Join with `races` to get the race date for chronological ordering.
3. Use a window function partitioned by `driverId`, ordered by race `date`, with `rowsBetween(unboundedPreceding, currentRow)` to compute the **cumulative sum** of each indicator.


In [0]:
# Step 1: indicator columns
df_q5_base = (
    df_results
    .select("raceId", "driverId", "positionOrder")
    .join(df_races.select("raceId", F.col("name").alias("race_name"), F.col("date").alias("race_date"), "year"), on="raceId")
    .join(
        df_drivers.select("driverId", F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("driver_name")),
        on="driverId"
    )
    .withColumn("is_win", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("is_2nd",  F.when(F.col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("is_3rd",  F.when(F.col("positionOrder") == 3, 1).otherwise(0))
)

# Step 2: cumulative window
w_driver_time = (
    Window
    .partitionBy("driverId")
    .orderBy("race_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_q5_result = (
    df_q5_base
    .withColumn("cumulative_wins",  F.sum("is_win").over(w_driver_time))
    .withColumn("cumulative_2nds",  F.sum("is_2nd").over(w_driver_time))
    .withColumn("cumulative_3rds",  F.sum("is_3rd").over(w_driver_time))
    .withColumn("cumulative_podiums", F.col("cumulative_wins") + F.col("cumulative_2nds") + F.col("cumulative_3rds"))
    .select(
        "raceId", "race_name", "race_date", "year",
        "driverId", "driver_name", "positionOrder",
        "cumulative_wins", "cumulative_2nds", "cumulative_3rds", "cumulative_podiums"
    )
    .orderBy("driverId", "race_date")
)

df_q5_result.show(30, truncate=False)


### Code Explanation — Q5

1. **`F.when(F.col("positionOrder") == 1, 1).otherwise(0)`** creates binary indicator columns: `is_win`, `is_2nd`, `is_3rd`. Each row gets a 1 only if the driver finished in that exact position.
2. **`Window.partitionBy("driverId").orderBy("race_date").rowsBetween(unboundedPreceding, currentRow)`** defines a growing window for each driver, ordered chronologically. This means for any given race, I look at all previous races plus the current one.
3. **`F.sum("is_win").over(w_driver_time)`** computes the running total of wins from the driver's first race up to the current race. The same pattern applies for 2nd and 3rd places.
4. **`cumulative_podiums`** is simply the sum of the three cumulative columns — giving the total podium count at any point in a driver's career.



### Extra Credit — Filter to Show Podium Milestones

In [0]:
# EXTRA CREDIT: show cumulative podiums only for drivers with 10+ podiums, at every 10th milestone
df_q5_ec = (
    df_q5_result
    .filter(F.col("cumulative_podiums") >= 10)
    .filter(F.col("cumulative_podiums") % 10 == 0)
    .select("driver_name", "race_name", "year", "cumulative_wins", "cumulative_2nds", "cumulative_3rds", "cumulative_podiums")
    .orderBy("driver_name", "cumulative_podiums")
)

print("=== Podium milestones (every 10th) for drivers with 10+ podiums ===")
df_q5_ec.show(30, truncate=False)


---
## Question 6 — Custom Question: Which Constructors Have the Most Consistent Pit Stops? [10 pts]

### Logic
**My Question:** Which constructors have the most consistent (lowest variance) pit stop times, and how does consistency relate to points scored?

Pit stop consistency is a proxy for how well-organized a team's pit crew is. I will:
1. Join `pit_stops` → `results` (to get `constructorId`), then group by `constructorId` to compute pit stop statistics (mean, stddev, min, count).
2. **Separately** aggregate total points from `results` by `constructorId` — this must be done independently because a driver can have multiple pit stops per race, and joining before aggregating would duplicate the `points` value for each pit stop row, inflating the total.
3. Join the two aggregated tables together on `constructorId`.
4. Compute the **coefficient of variation** (stddev / mean) as a normalized consistency metric.
5. Filter to constructors with at least 50 pit stops and rank by CV.

In [0]:
# 1) Pit stop statistics per constructor (each pit stop row is independent, no duplication issue)
df_pit_stats = (
    df_pit_stops
    .join(df_results.select("raceId", "driverId", "constructorId"), on=["raceId", "driverId"], how="inner")
    .groupBy("constructorId")
    .agg(
        F.round(F.avg("milliseconds") / 1000, 3).alias("avg_pit_sec"),
        F.round(F.stddev("milliseconds") / 1000, 3).alias("stddev_pit_sec"),
        F.round(F.min("milliseconds") / 1000, 3).alias("fastest_pit_sec"),
        F.count("*").alias("total_pit_stops")
    )
)

# 2) Points aggregated directly from results (one row per driver per race, no duplication)
df_points_stats = (
    df_results
    .groupBy("constructorId")
    .agg(F.round(F.sum("points"), 1).alias("total_points"))
)

# 3) Join the two aggregated tables
df_q6_result = (
    df_pit_stats
    .join(df_points_stats, on="constructorId", how="inner")
    .withColumn(
        "coeff_of_variation",
        F.round(F.col("stddev_pit_sec") / F.col("avg_pit_sec"), 4)
    )
    .filter(F.col("total_pit_stops") >= 50)
    .orderBy("coeff_of_variation")
)

print("=== Most Consistent Constructors (lowest CV, min 50 pit stops) ===")
df_q6_result.show(20, truncate=False)

### Code Explanation — Q6

1. **`df_pit_stops.join(df_results.select(..., "constructorId"), ...)`** attaches `constructorId` to each pit stop record. At this stage each row is still one pit stop, so aggregating `milliseconds` here is safe.
2. **`df_results.groupBy("constructorId").agg(F.sum("points"))`** computes total points **separately** from the pit stop table. This avoids the fan-out problem: if a driver has 3 pit stops in one race, joining before aggregating would triple-count that race's points.
3. **`.join(df_points_stats, on="constructorId")`** merges the two pre-aggregated tables, so each constructor has exactly one row with both pit stop stats and correct total points.
4. **Coefficient of Variation (`stddev / mean`)** normalizes the spread relative to the average, making it fair to compare constructors with different average pit times.
5. **`.filter(total_pit_stops >= 50)`** ensures I only look at constructors with a meaningful sample size.
6. **`.orderBy("coeff_of_variation")`** ranks constructors from most to least consistent.

This analysis can reveal whether more consistent pit crews correlate with higher total points — a potential indicator that pit crew efficiency contributes to overall race performance.
